# Análise de Qualidade de Dados — Olist Orders

Este notebook tem como objetivo avaliar a qualidade dos dados da tabela `olist_orders_dataset.csv` utilizando a biblioteca **Pandera**.

A análise será baseada nas seguintes dimensões de qualidade:

- **Completude:** verifica se os campos obrigatórios estão preenchidos;
- **Unicidade:** verifica se identificadores que deveriam ser únicos não estão duplicados;
- **Validade:** verifica se os valores pertencem aos domínios esperados;
- **Consistência:** verifica se informações relacionadas entre si respeitam regras lógicas do processo.

Para isso, será definido um **contrato de qualidade** utilizando um `DataFrameSchema`.

In [15]:
import pandas as pd
import pandera.pandas as pa

orders = pd.read_csv(
    "../data/raw/olist_orders_dataset.csv"
)

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [19]:
print(f"Quantidade de registros: {orders.shape[0]}")
print(f"Quantidade de colunas: {orders.shape[1]}")

display(orders.head())
orders.info()

Quantidade de registros: 99441
Quantidade de colunas: 8


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


## 1. Preparação dos dados

As colunas que representam datas são inicialmente carregadas pelo Pandas como texto.

Antes de realizar as validações, essas colunas são convertidas para o tipo `datetime`.

O parâmetro `errors="coerce"` faz com que valores que não possam ser interpretados como datas sejam convertidos para `NaT`, permitindo que sejam posteriormente identificados pelas regras de qualidade.

In [16]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

## 2. Definição do contrato de qualidade

O contrato de qualidade define as características esperadas para cada coluna da tabela.

As regras utilizadas avaliam:

### Unicidade

Como a granularidade da tabela é **1 linha = 1 pedido**, cada `order_id` deve ser único.

### Completude

Campos essenciais, como identificador do pedido, cliente, status, data da compra e previsão de entrega, não podem ser nulos.

Algumas datas do processo logístico permitem valores ausentes, pois isso pode ocorrer naturalmente dependendo do status do pedido. Por exemplo, um pedido cancelado pode não possuir data de entrega.

### Validade

O `order_status` deve pertencer ao conjunto de status conhecidos do processo da Olist.

### Consistência

Caso exista uma data de aprovação, ela não deve ocorrer antes da data em que a compra foi realizada.

In [23]:
orders_schema = pa.DataFrameSchema(
    {

        # -------------------------------------------------
        # UNICIDADE + COMPLETUDE
        # -------------------------------------------------
        # 1 linha = 1 pedido.
        # Portanto, order_id deve existir e ser único.
        "order_id": pa.Column(
            str,
            nullable=False,
            unique=True
        ),


        # -------------------------------------------------
        # COMPLETUDE
        # -------------------------------------------------
        # Todo pedido deve estar associado a um cliente.
        "customer_id": pa.Column(
            str,
            nullable=False
        ),


        # -------------------------------------------------
        # VALIDADE
        # -------------------------------------------------
        # O status deve pertencer ao conjunto esperado.
        "order_status": pa.Column(
            str,
            pa.Check.isin([
                "created",
                "approved",
                "invoiced",
                "processing",
                "shipped",
                "delivered",
                "unavailable",
                "canceled",
            ]),
            nullable=False
        ),


        # -------------------------------------------------
        # COMPLETUDE
        # -------------------------------------------------
        # Todo pedido deve possuir uma data de compra.
        "order_purchase_timestamp": pa.Column(
            "datetime64[ns]",
            nullable=False
        ),


        # A aprovação pode estar ausente dependendo
        # do estado do pedido.
        "order_approved_at": pa.Column(
            "datetime64[ns]",
            nullable=True
        ),


        # Nem todo pedido necessariamente chegou
        # à transportadora.
        "order_delivered_carrier_date": pa.Column(
            "datetime64[ns]",
            nullable=True
        ),


        # Nem todo pedido necessariamente foi entregue.
        "order_delivered_customer_date": pa.Column(
            "datetime64[ns]",
            nullable=True
        ),


        # A previsão de entrega deve existir.
        "order_estimated_delivery_date": pa.Column(
            "datetime64[ns]",
            nullable=False
        ),
    },


    # -----------------------------------------------------
    # CONSISTÊNCIA ENTRE COLUNAS
    # -----------------------------------------------------
    checks=[

        # Caso exista uma data de aprovação,
        # ela não deve ocorrer antes da compra.
        pa.Check(
            lambda df:
                df["order_delivered_carrier_date"].isna()
                |
                df["order_approved_at"].isna()
                |
                (
                    df["order_delivered_carrier_date"]
                    >= df["order_approved_at"]
                ),
            error=(
                "order_delivered_carrier_date não pode ocorrer "
                "antes de order_approved_at"
            )
        ),
    ]
)

## 3. Validação dos dados

Com o contrato definido, o DataFrame é validado utilizando o método `validate()`.

O parâmetro `lazy=True` permite que a Pandera continue verificando o conjunto de dados mesmo após encontrar uma violação, reunindo os problemas encontrados antes de gerar o erro.

Caso nenhuma regra seja violada, o dataset é considerado válido em relação ao contrato definido.

Caso existam violações, os registros problemáticos são apresentados através de `failure_cases`.

In [22]:
try:
    validated_orders = orders_schema.validate(
        orders,
        lazy=True
    )

    print("Dados válidos!")

except pa_errors.SchemaErrors as err:
    print("Foram encontrados problemas de qualidade.")

    error_summary = (
        err.failure_cases
        .groupby(["check_number", "check"])
        .agg(
            registros_afetados=("index", "nunique")
        )
        .reset_index()
    )

    display(error_summary)

Dados válidos!


## Conclusão

A utilização da Pandera permitiu representar as expectativas de qualidade da tabela `orders` através de um contrato explícito.

Foram avaliadas regras relacionadas à completude, unicidade, validade e consistência temporal dos dados.

A validação demonstra que qualidade de dados não depende apenas da ausência de valores nulos, mas também do cumprimento das regras estruturais e lógicas esperadas para o processo representado.